[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/02_langgraph.ipynb)

# Part 2 — Graphs (LangGraph)

> **Control flow: you, in code.** The model fills in nodes; the graph decides what runs next.

The pressure that produces this pattern: you need a **guarantee**. Something must *always*
happen — verification, logging, a human sign-off — and "the model usually remembers to" is not
good enough.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, re
if 'google.colab' in sys.modules:
    %pip install -U -q google-genai langgraph

from google import genai
from google.genai import types as gtypes

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""

print(f"Gemini client ready (model={MODEL}).")

### 2.1 The syntax

You declare a typed state, nodes that transform it, and edges. A node returns a *partial* update
to the state; the graph merges it. The router is ordinary Python, which means the control flow is
testable, diffable, and reviewable.

### 2.2 The toy, as a graph

Same arithmetic task — but now an **independent check** runs before any answer is returned, and
the retry budget is enforced in Python rather than requested in English.

Notice what the model is *not* allowed to do: it cannot skip `verify`, and it cannot talk its way
into a fourth attempt.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class GraphState(TypedDict):
    value: float
    attempts: int
    solves: int
    verified: bool

def n_compute(s):
    """The 'agent' node: ask the model and parse a number out of the reply."""
    txt = llm_text("Reply with only a number, no words, no units.",
                   f"What is 13 * 47 + {8 + s['attempts']}? Reply with only the number.")
    m = re.search(r"-?\d+(?:\.\d+)?", txt)
    val = float(m.group()) if m else float('nan')
    return {"value": val, "solves": s["solves"] + 1}

def n_verify(s):
    """Independent check. Recomputes rather than trusting the claim."""
    return {"verified": bool(abs(s["value"] - (13 * 47 + 8)) < 1e-9)}

def n_repair(s):
    return {"attempts": s["attempts"] + 1}

MAX_ATTEMPTS = 3
def route(s):
    if s["verified"]:                 return END
    if s["attempts"] >= MAX_ATTEMPTS: return END      # bounded in code, not in the prompt
    return "repair"

def build_graph(compute_fn):
    g = StateGraph(GraphState)
    g.add_node("compute", compute_fn)
    g.add_node("verify", n_verify)
    g.add_node("repair", n_repair)
    g.set_entry_point("compute")
    g.add_edge("compute", "verify")
    g.add_conditional_edges("verify", route, {"repair": "repair", END: END})
    g.add_edge("repair", "compute")
    return g.compile()

def run_graph(app, state):
    """Stream node-by-node so we can print the path the graph actually took."""
    path = []
    for step in app.stream(state, stream_mode="updates"):
        for node, update in step.items():
            path.append(node)
            state = {**state, **update}
    return state, path

app = build_graph(n_compute)
final, path = run_graph(app, {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(path))
print(f"value={final['value']}  verified={final['verified']}  "
      f"attempts={final['attempts']}  solves={final['solves']}")

### 2.3 The guarantee, demonstrated

The point of a graph is what happens when things go *wrong*. Below, the compute node never
produces the right answer. The run still terminates, still bounded, and still reports honestly
that it failed — with no prompt asking it to.

In [ ]:
def n_stuck(s):
    return {"value": 42.0, "solves": s["solves"] + 1}

bad, bpath = run_graph(build_graph(n_stuck),
                       {"value": 0.0, "attempts": 0, "solves": 0, "verified": False})
print("path:", " -> ".join(bpath))
print(f"verified={bad['verified']}  attempts={bad['attempts']}  solves={bad['solves']}")
assert bad["attempts"] == MAX_ATTEMPTS and not bad["verified"]
print("\nBounded, verified-or-honest, and reproducible. No prompt could have promised that.")

### 2.4 When to reach for a graph

**Use it when** the control flow is a *requirement*, not a choice:

| Requirement | Who enforces it |
|---|---|
| Every reported result is verified | the graph, not the prompt |
| At most *N* attempts | the router, in Python |
| A failed check triggers repair, not a retry | a conditional edge |
| The run is reproducible from a checkpoint | typed state |

**Its exemplar** is the production FEM workflow: a result that goes into a report has to have
been verified, every time, whatever the model felt like doing that morning.

**The cost:** you now maintain a graph. If your control flow genuinely is "whatever seems next,"
a graph is ceremony — use ReAct.